<a href="https://colab.research.google.com/github/zomweno/langchain/blob/master/Naive_Bayes_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Naive Bayes — Assignment



In [ ]:
import numpy as np
import pandas as pd
from collections import Counter

np.set_printoptions(precision=4, suppress=True)
RANDOM_STATE = 42


---
## Part 1 — Naive Bayes by hand (in code)

Below is a **different** tiny training set from the one in the lecture. Your job is
to compute, *from first principles* (no `sklearn` for this part), the quantities a
Multinomial Naive Bayes classifier would learn, then classify a new message.

**Training set (6 messages):**

| Message | Label |
|---|---|
| cheap loan offer now | SPAM |
| win cash prize now | SPAM |
| claim free loan today | SPAM |
| team lunch today | HAM |
| project review meeting | HAM |
| lunch with the team | HAM |

Use **Laplace smoothing with `alpha = 1`** and classify the message
**"free lunch"**.

$$P(w \mid y) = \frac{\text{count}(w, y) + \alpha}{\big(\sum_{w'}\text{count}(w', y)\big) + \alpha\,|V|}
\qquad
\hat{y} = \arg\max_y P(y)\prod_i P(w_i \mid y)$$


In [ ]:
train = [
    ("cheap loan offer now", "SPAM"),
    ("win cash prize now",   "SPAM"),
    ("claim free loan today","SPAM"),
    ("team lunch today",     "HAM"),
    ("project review meeting","HAM"),
    ("lunch with the team",  "HAM"),
]
test_message = "free lunch"
ALPHA = 1.0


**1a.** Compute the class priors `P(SPAM)` and `P(HAM)`. Store them in a dict
called `priors` keyed by the label string.

In [ ]:
# TODO 1a: compute priors as a dict, e.g. {"SPAM": ..., "HAM": ...}
n_total = len(train)
label_counts = Counter(label for _, label in train)
priors = {label: count / n_total for label, count in label_counts.items()}

# (leave this print so your output is visible when you submit)
print("Priors:", priors)


Priors: {'SPAM': 0.5, 'HAM': 0.5}


**1b.** Build the per-class word counts and compute the vocabulary size `V`
(number of *unique* words across the whole training set). Print the total token
count for each class and `V`.

In [ ]:
# TODO 1b: word_counts[label] -> Counter of word frequencies;
#          total_tokens[label] -> int; V -> int
word_counts = {"SPAM":Counter(), "HAM": Counter()}
for text, label in train:
  for word in text.split():
    word_counts[label][word] +=1

total_tokens = {label:sum(word_counts[label].values()) for label in word_counts}

all_words = set()
for counter in word_counts.values():
  all_words.update(counter.keys())
V = len(all_words)

print("V =", V)
print("total tokens:", total_tokens)


V = 17
total tokens: {'SPAM': 12, 'HAM': 10}


**1c.** Write a function `likelihood(word, label)` returning the smoothed
`P(word | label)`, then a function `score(message, label)` returning
`P(label) * product of likelihoods`. Use them to classify `test_message`.
Print each class score and the predicted label.

In [ ]:
# TODO 1c
def likelihood(word, label, alpha=ALPHA):
    return (word_counts[label][word] + alpha) / (total_tokens[label] + alpha * V)

def score(message, label, alpha=ALPHA):
    p = priors[label]
    for word in message.split():
      p *= likelihood(word, label, alpha)
    return p

scores = {label: score(test_message, label) for label in priors}  # TODO: fill with {label: score(test_message, label)}
print("Scores:", scores)
# print("Prediction:", ...)
print("Prediction:", max(scores, key=scores.get))


Scores: {'SPAM': 0.0011890606420927466, 'HAM': 0.0020576131687242796}
Prediction: HAM


**1d. (short answer)** The word *free* appears in SPAM but the word *lunch*
appears only in HAM. In one or two sentences, explain what add-1 smoothing does to
the likelihood of a word that has a **zero** count in a class, and why that matters
for this prediction. *(Double-click to edit, type your answer here.)*

> *Add-1 smoothing replaces every zero count with alpha, giving P(lunch | SPAM) = 1/29 instead of 0/12 = 0. Without smoothing, any word that never appeared in a class collapses the entire score for that class to zero; implying that the model could not predict SPAM for any message containing "lunch", regardless of any other evidence. Smoothing ensures all words get a small non-zero floor so every class stays in the running.*


---
## Part 2 — Build and tune a text classifier

Now scale up with `scikit-learn`. The cell below defines a small but realistic
product-review corpus labelled **positive** / **negative** (sentiment). It is
self-contained — no downloads.

Your task: build a `Pipeline`, tune it with `GridSearchCV` over the smoothing
strength **and** the n-gram range, then report **accuracy and macro-F1** on a
held-out test set.

In [ ]:
positive = [
    "absolutely love this it works perfectly and feels great",
    "best purchase this year I highly recommend it to everyone",
    "excellent and fast shipping I am very happy with it",
    "works exactly as described I would happily buy again",
    "fantastic it exceeded my expectations in every way",
    "super easy to set up and the battery lasts a long time",
    "great service and the item is wonderfully well made",
    "really impressed it feels premium solid and durable",
    "five stars it does everything I needed and more",
    "comfortable durable and it looks beautiful on my desk",
    "amazing value my whole family loves using it daily",
    "reliable and beautifully designed I have no complaints",
    "the materials feel premium and it works flawlessly",
    "shipping was quick and the packaging was lovely",
    "a perfect gift my friend was thrilled and delighted",
    "intuitive easy to use and it looks great everywhere",
    "outstanding performance and worth every penny spent",
    "so happy I bought this it works flawlessly every time",
    "helpful support and the product is excellent overall",
    "love the sturdy design comfortable and a joy to hold",
    "a great little device that does exactly what I wanted",
    "wonderful product fast delivery and a fair price",
    "highly satisfied I would definitely recommend it",
    "beautiful solid construction and a pleasure to use",
    "works like a charm and setup took just two minutes",
    "superb value comfortable reliable and well designed",
    "delighted with this excellent purchase works perfectly",
    "fantastic build I love it and use it every day happily",
    "great value durable and the support team was wonderful",
    "impressed by the premium feel and flawless performance",
]
negative = [
    "terrible it broke after just two days of careless use",
    "a waste of money it does not work as advertised",
    "very disappointed it feels cheap flimsy and broke fast",
    "it stopped working within a week I cannot recommend it",
    "awful experience and the support team never responded",
    "the worst purchase I have made it is completely useless",
    "poorly made it arrived damaged and I want a refund",
    "do not buy this it is overpriced unreliable and bad",
    "frustrating to use and the instructions were useless",
    "it failed on the first day total garbage honestly",
    "cheap flimsy materials and it broke almost immediately",
    "I regret buying this it never worked properly at all",
    "cheaply built it stopped charging after a few days",
    "it arrived broken and the support was useless and rude",
    "horrible it fell apart almost immediately so sad",
    "not worth it disappointed and I want my money back",
    "defective right out of the box and impossible to return",
    "slow flimsy and the worst service I have ever had",
    "complete junk it died after one week of light use",
    "misleading the product is nothing like the photos shown",
    "it broke the second time I used it terrible and cheap",
    "useless and overpriced I would avoid this seller",
    "terrible flimsy build and it stopped working fast",
    "deeply unhappy a total letdown and a waste of money",
    "flimsy plastic that cracked within days disappointed",
    "awful it broke immediately useless and overpriced junk",
    "the worst it failed instantly and support ignored me",
    "disappointed it is cheap unreliable and broke quickly",
    "terrible purchase useless flimsy and impossible to return",
    "horrible cheap and it stopped working after one use",
]
texts  = positive + negative
labels = ["pos"] * len(positive) + ["neg"] * len(negative)
print(pd.Series(labels).value_counts())


pos    30
neg    30
Name: count, dtype: int64


**2a.** Split the data into train/test (use `test_size=0.30`,
`random_state=RANDOM_STATE`, and `stratify=labels`).

In [ ]:
from sklearn.model_selection import train_test_split

# TODO 2a
X_train, X_test, y_train, y_test = train_test_split(
    texts, labels, test_size=0.30, random_state=RANDOM_STATE, stratify=labels
)
print(f"Train: {len(X_train)} Test: {len(X_test)}")

Train: 42 Test: 18


**2b.** Build a `Pipeline` of a vectorizer (`CountVectorizer` **or**
`TfidfVectorizer`, your choice) followed by `MultinomialNB`. Define a `param_grid`
that searches **at least**:
* the NB smoothing parameter `alpha` over several values, and
* the vectorizer `ngram_range` over `(1,1)` and `(1,2)`.

Run `GridSearchCV` with `scoring="f1_macro"` and `cv=4`. Print the best params and
best CV score.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import GridSearchCV

# TODO 2b
pipe = Pipeline([
    ("vec", TfidfVectorizer()),
    ("nb", MultinomialNB()),
])
param_grid = {
    "vec__ngram_range":[(1,1), (1,2)],
    "nb__alpha": [0.01, 0.1, 0.5, 1.0, 2.0, 5.0],
}
grid = GridSearchCV(pipe,param_grid, scoring="f1_macro", cv=4)

grid.fit(X_train, y_train)
print("Best params:", grid.best_params_)
print("Best CV macro-F1:", grid.best_score_)


Best params: {'nb__alpha': 0.01, 'vec__ngram_range': (1, 1)}
Best CV macro-F1: 0.7759837384837385


**2c.** Using the best estimator, predict on the test set and report both
**accuracy** and **macro-F1**. Also print the full `classification_report`.

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, classification_report

# TODO 2c
y_pred = grid.best_estimator_.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Macro-F1:", f1_score(y_test, y_pred, average="macro"))
print(classification_report(y_test, y_pred, digits=3))


Accuracy: 0.8888888888888888
Macro-F1: 0.8888888888888888
              precision    recall  f1-score   support

         neg      0.889     0.889     0.889         9
         pos      0.889     0.889     0.889         9

    accuracy                          0.889        18
   macro avg      0.889     0.889     0.889        18
weighted avg      0.889     0.889     0.889        18



**2d. (short answer)** Did adding bigrams (`ngram_range=(1,2)`) help, hurt, or
make no difference for your best model? Give one plausible reason, referring to the
size of this dataset. *(Double-click to edit.)*

> *Adding bigrams made no difference; the best model used unigrams only (ngram_range=(1,1)). With only 42 training reviews, most bigrams appear at most once, so that model has an almost no reliable frequency signal to learn from them. On a larger corpus, phrases like "not good" or "highly recommended" would carry meaningful sentiment that single words miss, but here they just inflate the vocabulary and add noise.*
